In [7]:
suppressPackageStartupMessages(library(tidyverse))
library(here)

options(box.path = here())
box::use(r / load)

r_out <- here("reports", "figures")
model_path <- here("data", "models")

PLOT_WIDTH <- 9
PLOT_HEIGHT <- 6


In [25]:
box::reload(load)
load$read_quantile_regs_from_folder(here(model_path, "views_elite", "afd"), "elite")

    lower_ci       est  upper_ci     rhat
1  0.2608220 0.3135247 0.3664776 1.000140
2  0.3641458 0.4385975 0.5086589 1.000169
3  0.4754170 0.5701107 0.6637202 1.000642
4  0.6072635 0.7203318 0.8320677 1.000069
5  0.7899944 0.9342240 1.0835263 1.000701
6  1.0548814 1.2164593 1.3820856 1.000160
7  1.4391184 1.6340591 1.8319326 1.000312
8  1.4234011 1.6299993 1.8401169 1.000587
9  1.4278364 1.6436319 1.8639884 1.000516
10 1.5735885 1.7963306 2.0275605 1.000602
11 1.7646052 1.9971657 2.2294003 1.000778
12 1.8238833 2.0717516 2.3288970 1.000323
13 2.1013033 2.4317160 2.7470035 1.000114
14 2.4757363 2.8691837 3.2528971 1.000551
15 3.3839277 3.8441402 4.3238532 1.000646
16 5.8639877 6.7629525 7.6162866 1.001195

In [ ]:
read_model <- function(model_type, party, model_file) {
    fpath <- here(model_path, model_type, party, model_file)
    readRDS(fpath)
}

create_summary <- function(model, var) {
    model_summary <- summary(model)

    df <- model_summary$fixed |>
        rownames_to_column("term") |>
        filter(term == !!var) |>
        rename(est = Estimate, upper_ci = `u-95% CI`, lower_ci = `l-95% CI`, rhat = Rhat) |>
        select(lower_ci, est, upper_ci, rhat)

    return(df)
}

read_files <- function() {
    model_types <- list.dirs(model_path, recursive = FALSE, full.names = FALSE)

    dfs <- list()
    i <- 0
    for (model_type in model_types) {
        words <- stringr::word(model_type, 1:2, sep = "_")
        type <- words[1]
        var <- words[2]

        parties <- list.dirs(here(model_path, model_type), recursive = FALSE, full.names = FALSE)

        for (party in parties) {
            model_files <- list.files(here(model_path, model_type, party), recursive = FALSE, full.names = FALSE)

            for (model_file in model_files) {
                quantile <- str_extract(model_file, "\\d+") |> as.integer()
                model <- read_model(model_type, party, model_file)

                i <- i + 1
                sub_df <- create_summary(model, var) |>
                    mutate(
                        party = party,
                        quantile = quantile,
                        type = type,
                        var = var
                    )
                dfs[[i]] <- sub_df
            }
        }
    }

    bind_rows(dfs)
}


In [ ]:
full_df <- read_files()


In [ ]:
cmap <- load$colormap()

party_name_map <- c(
    afd = "AfD",
    cdu_csu = "CDU/CSU",
    fdp = "FDP",
    greens = "Greens",
    left = "Left",
    spd = "SPD"
)

df <- full_df |>
    filter(quantile <= 95) |>
    mutate(party = recode(party, !!!party_name_map))


In [ ]:
model <- readRDS(here("data", "models", "views_elite", "cdu_csu", "q90.rds"))
summary(model)

In [ ]:
library(marginaleffects)

predictions <- marginaleffects::avg_predictions(
    model = model,
    by = c("elite", "channel"),
    transform = exp,
    newdata = marginaleffects::datagrid(
        model = model,
        elite = seq(0, 1, 0.05),
        grid_type = "counterfactual"
    )
)

predictions |>
    ggplot(aes(x = elite, y = estimate, color = channel)) +
    geom_line()


# Views Elite


In [ ]:
plot <- df |>
    filter(type == "views", var == "elite") |>
    ggplot(aes(x = quantile, y = est, color = party, fill = party)) +
    geom_line() +
    geom_ribbon(aes(ymin = lower_ci, ymax = upper_ci), linetype = 2, alpha = 0.2) +
    geom_hline(yintercept = 0, color = "black", linetype = "dashed") +
    facet_wrap(~party, scales = "fixed") +
    labs(
        x = "Quantile",
        y = "Coef. Anti-Elitism"
    ) +
    scale_color_manual(values = cmap) +
    scale_fill_manual(values = cmap) +
    theme_minimal(base_size = 14) +
    theme(legend.position = "none")

path <- here("reports", "figures", "reg_views_elite.svg")
if (file.exists(path)) file.remove(path)
ggsave(path, plot, width = PLOT_WIDTH, height = PLOT_HEIGHT)
plot


# Likes Elite


In [ ]:
plot <- df |>
    filter(type == "likes", var == "elite") |>
    ggplot(aes(x = quantile, y = est, color = party, fill = party)) +
    geom_line() +
    geom_ribbon(aes(ymin = lower_ci, ymax = upper_ci), linetype = 2, alpha = 0.2) +
    geom_hline(yintercept = 0, color = "black", linetype = "dashed") +
    facet_wrap(~party, scales = "fixed") +
    labs(
        x = "Quantile",
        y = "Coef. Anti-Elitism"
    ) +
    scale_color_manual(values = cmap) +
    scale_fill_manual(values = cmap) +
    theme_minimal(base_size = 14) +
    theme(legend.position = "none")

path <- here("reports", "figures", "reg_likes_elite.svg")
if (file.exists(path)) file.remove(path)
ggsave(path, plot, width = PLOT_WIDTH, height = PLOT_HEIGHT)
plot


# Views Pplcentr


In [ ]:
plot <- df |>
    filter(type == "views", var == "pplcentr") |>
    ggplot(aes(x = quantile, y = est, color = party, fill = party)) +
    geom_line() +
    geom_ribbon(aes(ymin = lower_ci, ymax = upper_ci), linetype = 2, alpha = 0.2) +
    geom_hline(yintercept = 0, color = "black", linetype = "dashed") +
    facet_wrap(~party, scales = "fixed") +
    labs(
        x = "Quantile",
        y = "Coef. People-Centrism"
    ) +
    scale_color_manual(values = cmap) +
    scale_fill_manual(values = cmap) +
    theme_minimal(base_size = 14) +
    theme(legend.position = "none")

path <- here("reports", "figures", "reg_views_pplcentr.svg")
if (file.exists(path)) file.remove(path)
ggsave(path, plot, width = PLOT_WIDTH, height = PLOT_HEIGHT)
plot


# Likes Pplcentr


In [ ]:
plot <- df |>
    filter(type == "likes", var == "pplcentr") |>
    ggplot(aes(x = quantile, y = est, color = party, fill = party)) +
    geom_line() +
    geom_ribbon(aes(ymin = lower_ci, ymax = upper_ci), linetype = 2, alpha = 0.2) +
    geom_hline(yintercept = 0, color = "black", linetype = "dashed") +
    facet_wrap(~party, scales = "fixed") +
    labs(
        x = "Quantile",
        y = "Coef. People-Centrism"
    ) +
    scale_color_manual(values = cmap) +
    scale_fill_manual(values = cmap) +
    theme_minimal(base_size = 14) +
    theme(legend.position = "none")

path <- here("reports", "figures", "reg_likes_pplcentr.svg")
if (file.exists(path)) file.remove(path)
ggsave(path, plot, width = PLOT_WIDTH, height = PLOT_HEIGHT)
plot


In [ ]:
min(df$rhat)
max(df$rhat)
